# Save logRR for all countries and expand grid at edges (central estimates only)

NaN values at the edge of curves are transformed to edge logRR

In [1]:
import re
import os
import json
import numpy as np
import pandas as pd
import xarray as xr
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === Path config ===
ID_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23")
# Load country names and ids
lookup_file = "country_location_ids.json"
lookup_path = os.path.join(ID_DIR, lookup_file)
with open(lookup_path, "r", encoding="utf-8") as f:
    country_names = json.load(f)  # {name: id}

In [3]:
# === Health variables ===
health_vars = ["CHRONIC_KIDNEY_DISEASE", "CARDIOMYOPATHY_MYOCARDITIS",
               "HYPERTENSIVE_HEART_DISEASE", "ISCHEMIC_HEART_DISEASE",
               "STROKE", "DIABETES", "INJ_ANIMAL", "INJ_DISASTER",
               "DROWNING", "HOMICIDE", "MECHANICAL_INJ",
               "OTHER_UNINTENTIONAL_INJ", "SUICIDE", "OTHER_TRANSPORT_INJ",
               "ROAD_TRANSPORT_INJ", "LOWER_RESPIRATORY_INFECTIONS", "COPD"]

In [4]:
def fill_edge_nans(da, dim="daily_temperature"):
    valid = da.notnull()

    # leading NaNs: no valid value anywhere before this point
    leading = valid.cumsum(dim) == 0

    # trailing NaNs: no valid value anywhere after this point
    rev = slice(None, None, -1)
    trailing = valid[{dim: rev}].cumsum(dim)[{dim: rev}] == 0

    # nearest valid value at each end
    filled = da.ffill(dim).bfill(dim)

    # use the filled values only at the edges, keep everything else as is
    return xr.where(leading | trailing, filled, da)

In [6]:
# Create file path dictionary to country specific ERFs
for health_VAR in health_vars:
    print(f"Processing {health_VAR}")

    DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23" / "RR_curves" / "temperature" / health_VAR)
    # filenames are exactly "<health_VAR>_norm_logRR_<id>.nc"
    country_files = {int(cid): DIR / f"{health_VAR}_norm_mean_logRR_{cid}.nc" for cid in country_names.values()}

    # Check which files exist and which IDs are missing
    missing = [cid for cid, p in country_files.items() if not p.exists()]
    print("IDs in dict with no file:", missing)
    country_files = {cid: p for cid, p in country_files.items() if p.exists()}

    # Files on disk that aren't in the dict
    on_disk = {int(re.search(r"\d+", p.stem).group()) for p in DIR.glob("*.nc")
               if re.search(r"\d+", p.stem)}
    print("Files with IDs not in dict:", on_disk - set(country_files) - set(missing))  # will be 1 because of global

    # Combine per-country files into one array with a 'country' dim
    logRR = xr.concat(
        [xr.open_dataarray(p) for p in country_files.values()],
        dim=pd.Index(list(country_files.keys()), name="country"),
        join="exact",  # errors if annual_temp/daily_temp grids differ between countries
    )

    # fill edge nans
    logRR_fill = fill_edge_nans(logRR)

    out_file = f"{health_VAR}_norm_mean_logRR.nc"
    out_path = os.path.join(DIR, out_file)
    print(f"Saving to {out_path}")
    logRR_fill.to_netcdf(out_path)

print("All processing complete.")

Processing CHRONIC_KIDNEY_DISEASE
IDs in dict with no file: []
Files with IDs not in dict: {1}
Saving to /glade/work/awells/workflow/GBD23/RR_curves/temperature/CHRONIC_KIDNEY_DISEASE/CHRONIC_KIDNEY_DISEASE_norm_mean_logRR.nc
Processing CARDIOMYOPATHY_MYOCARDITIS
IDs in dict with no file: []
Files with IDs not in dict: {1}
Saving to /glade/work/awells/workflow/GBD23/RR_curves/temperature/CARDIOMYOPATHY_MYOCARDITIS/CARDIOMYOPATHY_MYOCARDITIS_norm_mean_logRR.nc
Processing HYPERTENSIVE_HEART_DISEASE
IDs in dict with no file: []
Files with IDs not in dict: {1}
Saving to /glade/work/awells/workflow/GBD23/RR_curves/temperature/HYPERTENSIVE_HEART_DISEASE/HYPERTENSIVE_HEART_DISEASE_norm_mean_logRR.nc
Processing ISCHEMIC_HEART_DISEASE
IDs in dict with no file: []
Files with IDs not in dict: {1}
Saving to /glade/work/awells/workflow/GBD23/RR_curves/temperature/ISCHEMIC_HEART_DISEASE/ISCHEMIC_HEART_DISEASE_norm_mean_logRR.nc
Processing STROKE
IDs in dict with no file: []
Files with IDs not in dic